# Pipeline Inputs

In [1]:
IMAGE_PATH = "data/temp/X00016469612.jpg"
DOC_CATEGORY = "receipt"
FIELDS = ["company", "date", "address", "total", "phone number"]
DEBUG = True   

In [2]:
import os, gc, re, json, time
import torch
from PIL import Image, ImageDraw, ImageFont
import pytesseract
from transformers import AutoTokenizer, AutoModelForCausalLM
from datetime import datetime
import pandas as pd
import colorsys

/home/lu.dong1/.conda/envs/expo-judge/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Model Paths

In [3]:
pytesseract.pytesseract.tesseract_cmd = "/home/lu.dong1/.conda/envs/expo-judge/bin/tesseract"

ENGINE_A_PATH = "/projects/insightx-lab/cn_grpo/models/Llama-3.1-8B-Instruct"
ENGINE_B_PATH = "/projects/insightx-lab/cn_grpo/models/Qwen2.5-7B-Instruct"
# EVAL_MODEL_PATH = "/projects/insightx-lab/cn_grpo/models/Llama-3.1-8B-Instruct"
EVAL_MODEL_PATH = "/projects/insightx-lab/cn_grpo/models/Mistral-7B-Instruct-v0.3"

# Debug output directory (only used when DEBUG=True)
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
DEBUG_OUTPUT_DIR = f"data/temp/debug/{RUN_TS}"

In [4]:
def generate_field_colors(n: int):
    colors = []
    
    for i in range(n):
        hue = (i / n + 0.55) % 1.0
        r, g, b = colorsys.hsv_to_rgb(hue, 0.65, 0.90)
        colors.append((int(r * 255), int(g * 255), int(b * 255)))
        
    return colors

In [5]:
STATE_BORDER = {
    "pass": (34,  197,  94),   # green
    "review_needed": (251, 146,  60),   # orange
    "fail": (239,  68,  68),   # red
}

STATE_ICON = {
    "pass": "✓",
    "review_needed": "⚠",
    "fail": "✗",
}

# Common Utils

In [6]:
def save_text(text, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

In [7]:
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

In [8]:
def maybe_save_text(text, filename, debug, output_dir):
    if debug:
        os.makedirs(output_dir, exist_ok=True)
        save_text(text, os.path.join(output_dir, filename))

In [9]:
def maybe_save_json(obj, filename, debug, output_dir):
    if debug:
        os.makedirs(output_dir, exist_ok=True)
        save_json(obj, os.path.join(output_dir, filename))

In [10]:
def try_parse_json(text):
    for fn in [lambda t: json.loads(t),
               lambda t: json.loads(extract_json_block(t))]:
        try:
            return fn(text)
        except Exception:
            pass
    return None

In [11]:
def extract_json_block(text: str) -> str:
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return match.group(0)
    return text

In [12]:
def load_model_and_tokenizer(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    return tokenizer, model

In [13]:
def run_chat_inference(model, tokenizer, prompt: str, max_new_tokens: int = 1200, return_usage: bool = False):
    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    prompt_tokens = int(inputs["input_ids"].shape[1])
    started_at = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    elapsed_seconds = round(time.time() - started_at, 4)
    completion_tokens = int(outputs[0].shape[0] - inputs["input_ids"].shape[1])

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    if not return_usage:
        return response

    usage = {
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": prompt_tokens + completion_tokens,
        "elapsed_seconds": elapsed_seconds,
    }
    return response, usage

In [14]:
def release_model(model=None, tokenizer=None):
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Prompt Builders

In [15]:
def _fields_str(fields):
    """Return a bullet list string for prompt injection."""
    return "\n".join(f"- {f}" for f in fields)

In [16]:
def _ocr_alignment_guideline():
    return """
OCR alignment measures whether the extracted value is grounded in OCR raw text with the correct local semantic context for the target field.

Do NOT judge this only by whether the value appears somewhere in the OCR text.

A value may appear in OCR text but still have weak OCR alignment if:
- it belongs to a different field,
- it appears under a different nearby label,
- it comes from a different section of the document,
- it is only partially matched,
- or the surrounding OCR text does not support that it corresponds to the target field.

Use the following scale:
- strong: the extracted value is clearly supported by OCR text, and the nearby context strongly indicates that it belongs to the target field.
- partial: the extracted value is partly supported by OCR text, but the context is incomplete, ambiguous, noisy, or only indirectly related to the target field.
- weak: the extracted value is unsupported, mismatched, only coincidentally present, or grounded in the wrong context.
""".strip()

### Stage-2A: Constraints Generation

In [17]:
def build_constraints_prompt_A(raw_text, doc_category, fields):
    return f"""
You are a constraint inference engine for OCR-based information extraction.

Document category:
{doc_category}

Target fields:
{_fields_str(fields)}

OCR raw text:
{raw_text}

Your task:
Infer GENERAL validation constraints for each target field based on:
1. the document category
2. the OCR text patterns visible in this document
3. the likely semantic role of each field

Important:
These constraints are NOT the extracted field values.
They are reusable field-level validation hints that can later help evaluate whether a candidate value is credible.

Think at the level of:
- semantic type
- common text form
- likely context signals
- broad structural tendencies in text

Examples of useful constraint types:
- whether a field usually looks like a name, label, amount, date, identifier, short phrase, or multi-line text block
- whether a field tends to have numeric, alphabetic, mixed, or symbolic characters
- whether a field often appears with nearby cue words or local context
- whether a field is usually short, medium, or multi-line
- whether a field may contain delimiters, currency symbols, punctuation, or formatting markers

Rules:
- Be general and reusable.
- Do NOT copy exact values from the OCR text as constraints.
- Do NOT invent document-specific facts.
- Do NOT rely on exact position or page layout assumptions such as top, bottom, left, or right.
- Do NOT require strict regex-like rules unless they are truly fundamental.
- Prefer soft validation guidance over brittle hard rules.
- Keep each field's constraints concise and practical.

Return ONLY valid JSON in exactly this format:

{{
  "constraint_summary": {{
    {", ".join([f'"{field}": ["constraint 1", "constraint 2"]' for field in fields])}
  }}
}}
""".strip()

In [18]:
def build_constraints_prompt_B(raw_text, doc_category, fields):
    return f"""
You are a constraint inference engine for OCR-based information extraction.

Document category:
{doc_category}

Target fields:
{_fields_str(fields)}

OCR raw text:
{raw_text}

Your task:
Infer GENERAL validation constraints for each target field based on:
1. the document category
2. the OCR text patterns visible in this document
3. the likely semantic role of each field

Important:
These constraints are NOT the extracted field values.
They are reusable field-level validation hints that can later help evaluate whether a candidate value is credible.

Think at the level of:
- semantic type
- common text form
- likely context signals
- broad structural tendencies in text

Examples of useful constraint types:
- whether a field usually looks like a name, label, amount, date, identifier, short phrase, or multi-line text block
- whether a field tends to have numeric, alphabetic, mixed, or symbolic characters
- whether a field often appears with nearby cue words or local context
- whether a field is usually short, medium, or multi-line
- whether a field may contain delimiters, currency symbols, punctuation, or formatting markers

Rules:
- Be general and reusable.
- Do NOT copy exact values from the OCR text as constraints.
- Do NOT invent document-specific facts.
- Do NOT rely on exact position or page layout assumptions such as top, bottom, left, or right.
- Do NOT require strict regex-like rules unless they are truly fundamental.
- Prefer soft validation guidance over brittle hard rules.
- Keep each field's constraints concise and practical.

Return ONLY valid JSON in exactly this format:

{{
  "constraint_summary": {{
    {", ".join([f'"{field}": ["constraint 1", "constraint 2"]' for field in fields])}
  }}
}}
""".strip()

### Stage-2B: Constraint-guided Extraction

In [19]:
def build_extraction_prompt(raw_text, constraint_trace, fields, doc_category):
    field_extraction_block = ",\n".join([f'    "{f}": ""' for f in fields])
    evidence_block = ",\n".join([f'    "{f}": ""' for f in fields])
    reasoning_block = ",\n".join([f'    "{f}": ""' for f in fields])

    return f"""
You are an information extraction engine for {doc_category} documents.

You are given:
1. OCR raw text from a {doc_category}
2. A constraint trace previously generated for this document

Your task:
Extract the following fields:
{_fields_str(fields)}

For each field, provide:
1. field_extraction: the extracted value
2. evidence_trace: the most directly relevant supporting OCR text
3. reasoning: explain why this evidence supports the extraction and how the constraint trace helped

Important guidelines:
- Extract each field independently.
- Use only the OCR raw text and the constraint trace below.
- Do NOT use outside knowledge.
- Every field listed in the schema MUST appear in the output, even if no value is found.
- If a field is partially visible or uncertain but still plausibly present, provide the best supported guess and mention the uncertainty in reasoning.
- If a field is not present in the OCR text, or no sufficiently supported candidate can be identified, you MUST return an empty string "" for that field. Do NOT omit the field.
- Do NOT hallucinate a value just to fill the schema.
- The evidence_trace for each field should be SHORT, LOCAL, and DIRECTLY GROUNDED in the OCR text.
- Do NOT use large unrelated text blocks as evidence.
- Do NOT reuse the same generic evidence for multiple fields unless it clearly supports each field.
- The extracted value should be supported by the evidence_trace for that field.
- If field_extraction for a field is "", then evidence_trace for that field must also be "", and reasoning should briefly explain that no sufficiently supported value was found.

Special rule for OCR noise:
- If the selected field value or supporting OCR evidence contains "?", preserve it exactly as-is.
- Do NOT remove, replace, or normalize question marks.
- Do NOT guess missing characters hidden by "?".

Schema completeness rules:
- You MUST return every field listed in the schema.
- Do NOT omit any field, even if the field is not present in the document.
- If a field is missing or not found, set:
  - field_extraction[field] = ""
  - evidence_trace[field] = ""
  - reasoning[field] = "No sufficiently supported value was found for this field."
- Missing fields must still appear in the JSON output.

Return ONLY valid JSON in exactly this format:

{{
  "field_extraction": {{
{field_extraction_block}
  }},
  "evidence_trace": {{
{evidence_block}
  }},
  "reasoning": {{
{reasoning_block}
  }}
}}

Constraint trace:
{constraint_trace}

OCR raw text:
{raw_text}
""".strip()

In [20]:
def build_single_extraction_prompt(raw_text, constraint_trace, field, doc_category):
    return f"""
You are an information extraction engine for {doc_category} documents.

You are given:
1. OCR raw text from a {doc_category}
2. A constraint trace previously generated for this document

Your task:
Extract ONLY this field:
- {field}

For this field, provide:
1. field_extraction: the extracted value
2. evidence_trace: the most directly relevant supporting OCR text
3. reasoning: explain why this evidence supports the extraction and how the constraint trace helped

Important guidelines:
- Use only the OCR raw text and the constraint trace below.
- Do NOT use outside knowledge.
- If the field is partially visible or uncertain but still plausibly present, provide the best supported guess and mention the uncertainty in reasoning.
- If the field is not present in the OCR text, or no sufficiently supported candidate can be identified, you MUST return an empty string "" for field_extraction.
- Do NOT hallucinate a value.
- The evidence_trace should be SHORT, LOCAL, and DIRECTLY GROUNDED in the OCR text.
- Do NOT use large unrelated text blocks as evidence.
- The extracted value should be supported by the evidence_trace.
- If field_extraction is "", then evidence_trace must also be "", and reasoning should briefly explain that no sufficiently supported value was found.

Special rule for OCR noise:
- If the selected field value or supporting OCR evidence contains "?", preserve it exactly as-is.
- Do NOT remove, replace, or normalize question marks.
- Do NOT guess missing characters hidden by "?".

Return ONLY valid JSON in exactly this format:

{{
  "field_extraction": "",
  "evidence_trace": "",
  "reasoning": ""
}}

Constraint trace:
{constraint_trace}

OCR raw text:
{raw_text}
""".strip()

### Stage-3A: Rule Synthesis

In [21]:
def build_rule_synthesis_prompt(engineA_constraints_text, engineB_constraints_text, doc_category, fields):
    consolidated_block = "\n".join(
        f'    "{f}": [],' for f in fields[:-1]
    ) + f'\n    "{fields[-1]}": []'

    return f"""
Return ONLY valid JSON.
Do not output explanations.

You are a rule synthesis engine for {doc_category} field evaluation.

Two extraction engines independently analyzed the OCR text and produced
field-level validation constraints.

Your task is to synthesize them into ONE shared set of consolidated checking rules
for evaluating the credibility of extracted fields.

Fields:
{_fields_str(fields)}

These rules will later be used for self-justification and cross-engine judgment.

--------------------------------
SYNTHESIS PRINCIPLES
--------------------------------
1. Do NOT simply merge or concatenate rules from both engines.
2. If two rules express similar ideas, merge them into ONE generalized rule.
3. Prefer generalized plausibility rules rather than strict schema validation.
4. Avoid overly brittle rules that depend on a specific OCR instance.
5. Preserve useful signals such as format clues, semantic meaning, local contextual clues in text, and numeric or textual patterns.
6. The goal is evaluation-oriented checking rules, not strict regex validation.
7. Each field should contain 3–5 compact checking rules.
8. synthesis_summary.notes should briefly summarize the main agreement or disagreement patterns between the two engines.
9. Keep synthesis_summary.notes short, concrete, and limited to one sentence.
10. Do not leave synthesis_summary.notes empty unless both engine outputs are completely uninformative.

--------------------------------
GOOD RULE EXAMPLES
--------------------------------
Good:
"should resemble a plausible value for the target field"
"may follow a common text or numeric pattern associated with the field"
"should match the expected semantic role and local text context of the field"

Bad:
"must exactly match regex ^...$"
"must appear in a fixed document position"
"must satisfy a rigid maximum length constraint"

--------------------------------
OUTPUT FORMAT
--------------------------------
Return JSON in EXACTLY this format:
{{
  "consolidated_rules": {{
{consolidated_block}
  }},
  "synthesis_summary": {{
    "agreement_level": "high | moderate | low",
    "notes": ""
  }}
}}

--------------------------------
ENGINE A OUTPUT
--------------------------------
{engineA_constraints_text}

--------------------------------
ENGINE B OUTPUT
--------------------------------
{engineB_constraints_text}
""".strip()

### Stage-3B: Self-Justification

In [22]:
def build_stage2_prompt(
    field_name,
    rules,
    extracted_value,
    evidence_text,
    reasoning_text,
    ocr_raw_text,
    doc_category
):
    return f"""
You are an evaluation model for {doc_category} field extraction reliability.

Your job is to evaluate whether the extracted value for one field is credible.

Return ONLY valid JSON. No explanation.

Field name:
{field_name}

Consolidated validation rules:
{json.dumps(rules, indent=2, ensure_ascii=False)}

Extracted value:
{extracted_value}

Evidence trace:
{evidence_text}

Engine reasoning:
{reasoning_text}

OCR raw text:
{ocr_raw_text}

Evaluation definitions:
1. rule_consistency:
How well the extracted value matches the consolidated validation rules for this field.

2. engine_self_consistency:
How well the engine's evidence trace and reasoning support its own extracted value.

3. ocr_alignment:
How well the extracted value is grounded in OCR raw text with the correct local semantic context for the target field.

Use the following scale:
- strong: the value is clearly grounded in OCR text, and the surrounding OCR context strongly supports that it belongs to the target field.
- partial: the value has some OCR support, but the surrounding context is incomplete, ambiguous, noisy, or only indirectly supportive.
- weak: the value is ungrounded, mismatched, coincidental, or supported by the wrong context.

4. ocr_corruption:
Whether the extracted value itself appears to contain OCR corruption.

Judge OCR corruption mainly from the extracted value string itself.
Focus on visible recognition artifacts in the text form, not just on whether the overall meaning seems plausible.

Use the following scale:
- absent: no obvious OCR corruption is present; the value appears clean, readable, and textually well-formed.
- possible: mild or uncertain OCR corruption may be present; the value is still partly readable, but some segments, characters, or symbols are questionable.
- present: obvious OCR corruption is present; the value contains clear recognition artifacts such as unexpected "?" characters, broken segments, unusual symbol substitutions, merged fragments, or corrupted text that makes the semantic meaning unclear.

General hints:
- Unexpected "?" characters inside a word, number, or phrase are strong evidence of OCR corruption.
- Strange symbol substitutions, broken fragments, or malformed character sequences may indicate OCR corruption even if part of the value is still readable.
- A value can still be semantically plausible while being OCR-corrupted.
- Clean punctuation alone does not imply corruption.
- Normal delimiters such as commas, periods, slashes, hyphens, ampersands, or parentheses are not OCR corruption by themselves.

Examples:
- absent:
  "2023-08-15"
  "123 Main Street"
  "Total: 45.90"
- possible:
  "2O23-08-15"
  "123 Main Stree1"
  "Tota1: 45.90"
- present:
  "2O?3-0?15"
  "123 Ma?n Str?et"
  "To?al: 4?.9O"

Important:
- OCR corruption does NOT automatically mean the value is wrong.
- A value may still be partially usable even if OCR corruption is present.
- Focus on whether corruption is absent, possible, or clearly present.

5. judgment_summary:
Write one short summary sentence explaining the overall credibility of the extracted value.
The summary must be consistent with the other output fields.
Do not introduce new evidence or new conclusions beyond:
- rule_consistency
- engine_self_consistency
- ocr_alignment
- ocr_corruption

Return JSON in exactly this format:

{{
  "extracted_value": "",
  "rule_consistency": "high | moderate | low",
  "engine_self_consistency": "strong | moderate | weak",
  "ocr_alignment": "strong | partial | weak",
  "ocr_corruption": "absent | possible | present",
  "judgment_summary": ""
}}
""".strip()

### Stage-3C: Cross-Judgment

In [23]:
def build_cross_judge_prompt(
    field_name,
    consolidated_rules_for_field,
    engineA_field_result,
    engineB_field_result,
    doc_category
):
    return f"""
Return ONLY valid JSON.
Do not output explanations.

You are a cross-engine judgment model for {doc_category} field extraction reliability.

Your task is to compare the self-justification results from engineA and engineB
for one field, and decide which extracted value is more credible.

Field name:
{field_name}

Consolidated checking rules:
{json.dumps(consolidated_rules_for_field, indent=2, ensure_ascii=False)}

EngineA result:
{json.dumps(engineA_field_result, indent=2, ensure_ascii=False)}

EngineB result:
{json.dumps(engineB_field_result, indent=2, ensure_ascii=False)}

Judgment instructions:
1. Compare the two candidate extractions and their stage-2 signals.
2. Select the more credible extracted value.
3. selected_engine must be exactly one of:
   - "engineA"
   - "engineB"
   - "none"
4. Use selected_engine = "none" only if neither engine provides a usable recommendation.
5. If selected_engine = "none", recommended_value must be an empty string.
6. Treat extraction quality as the primary factor.
7. Use engine efficiency only as a secondary tiebreaker when both candidates are similarly credible or both are clearly good.
8. Engine efficiency is reflected by lower total_tokens and lower elapsed_seconds in the engine_metrics metadata.
9. Do NOT prefer a less credible extraction just because it is faster or cheaper.

Field confidence definition:
field_confidence should reflect the overall credibility of the recommended value,
based on:
- final_rule_consistency
- final_engine_self_consistency
- final_ocr_alignment
- final_ocr_corruption

Use the following scale:
- very_high: all major support signals are very strong, and OCR corruption is absent.
- high: support signals are strong overall, and OCR corruption is absent.
- medium: the value is still plausible, but one or more support signals is not strong,
  or OCR corruption is possible.
- low: the value is weakly supported, OCR corruption is present, or no usable value can be recommended.

Field state definitions:
- pass:
  use this only when OCR corruption is absent and the three main support signals
  (rule consistency, engine self-consistency, OCR alignment) are all strong.
- review_needed:
  use this when a usable candidate still exists, but OCR corruption is possible or present,
  or one or more of the three support signals is not strong.
- fail:
  use this when no usable candidate value is available,
  or when the three support signals are collectively too weak to justify a credible recommendation.

Important clarification:
- OCR corruption means recognition artifacts such as unexpected "?" characters,
  unusual symbol substitutions, broken segments, or corrupted text that makes the semantic meaning unclear.
- OCR corruption does NOT automatically mean fail.
- A non-empty but OCR-corrupted value should usually be labeled review_needed, not fail.
- Example: "NO.5? $5,57 & 59, JALAN SAGU 18" is OCR-corrupted text and should usually be treated as review_needed if it still provides a partially usable candidate.

State reason:
- state_reason explains why the final field_state is "review_needed" or "fail".
- If field_state = "pass", state_reason must be an empty string.
- If field_state = "review_needed" or "fail", state_reason must be a short, specific reason.

Decision rules:
- If selected_engine = "none" or recommended_value is empty, use field_state = "fail".
- If the recommended value is non-empty but OCR corruption is possible or present, use field_state = "review_needed".
- If OCR corruption is absent, and all three main support signals are strong, use field_state = "pass".
- If OCR corruption is absent, but one or more of the three support signals is not strong, use field_state = "review_needed".
- Use fail only when no usable recommendation exists, or the support signals are collectively too weak.

Consistency constraints:
- If field_state = "pass", field_confidence should be "high" or "very_high".
- If field_state = "review_needed", field_confidence should be "medium" or "low".
- If field_state = "fail", field_confidence must be "low".
- Do NOT output multiple engines.
- Do NOT output "engineA | engineB".

Return JSON in exactly this format:
{{
  "recommended_value": "",
  "selected_engine": "",
  "selection_reason": "",
  "final_rule_consistency": "high | moderate | low",
  "final_engine_self_consistency": "strong | moderate | weak",
  "final_ocr_alignment": "strong | partial | weak",
  "final_ocr_corruption": "absent | possible | present",
  "field_confidence": "very_high | high | medium | low",
  "field_state": "pass | review_needed | fail",
  "state_reason": ""
}}
""".strip()

# Pipeline Stages

In [24]:
# def run_sanity_check(value):
#     value = str(value).strip() if value is not None else ""
#     if not value or "?" in value:
#         return "not_pass"
#     return "pass"

In [25]:
def run_sanity_check(value):
    value = str(value).strip() if value is not None else ""
    if not value:
        return "not_pass"
    return "pass"

In [26]:
def normalize_extraction_output(extraction, fields):
    extraction = extraction or {}

    field_extraction = extraction.get("field_extraction", {})
    evidence_trace = extraction.get("evidence_trace", {})
    reasoning = extraction.get("reasoning", {})

    for field in fields:
        if field not in field_extraction:
            field_extraction[field] = ""
        if field not in evidence_trace:
            evidence_trace[field] = ""
        if field not in reasoning:
            reasoning[field] = "No sufficiently supported value was found for this field."

    extraction["field_extraction"] = field_extraction
    extraction["evidence_trace"] = evidence_trace
    extraction["reasoning"] = reasoning
    return extraction

In [27]:
def match_evidence_to_bboxes(evidence_text: str, word_df: pd.DataFrame):
    """
    Find the most matching consecutive word sequence in word_df for the given evidence_text,
    and return a merged bounding box (left, top, right, bottom).
    """
    if not evidence_text or not evidence_text.strip():
        return None

    # Normalize evidence and OCR words
    def clean(s):
        return re.sub(r"[^\w\s]", "", s.lower()).split()

    ev_words = clean(evidence_text)
    ocr_words = [clean(str(w)) for w in word_df["text"].tolist()]
    # ocr_words[i] might be empty (pure punctuation removed), fallback to original word
    ocr_words_flat = [
        (w[0] if w else str(word_df["text"].iloc[i]).lower())
        for i, w in enumerate(ocr_words)
    ]

    if not ev_words:
        return None

    best_score = 0
    best_span = None
    n = len(ocr_words_flat)
    m = len(ev_words)

    # Sliding window find the consecutive segment with the highest overlap score
    for i in range(n - m + 1):
        window = ocr_words_flat[i:i + m]
        score = sum(1 for a, b in zip(ev_words, window) if a == b)
        if score > best_score:
            best_score = score
            best_span = (i, i + m - 1)

    # if score is too low discard
    if best_score < max(1, len(ev_words) * 0.4):
        return None

    # Merge bounding boxes for all words in the best span
    rows = word_df.iloc[best_span[0]: best_span[1] + 1]
    left = int(rows["left"].min())
    top = int(rows["top"].min())
    right = int((rows["left"] + rows["width"]).max())
    bottom = int((rows["top"]  + rows["height"]).max())
    return (left, top, right, bottom)


In [28]:
# def phase_ocr(image_path):
#     image = Image.open(image_path)
#     return pytesseract.image_to_string(image)

In [29]:
def phase_ocr_with_bbox(image_path):
    image = Image.open(image_path)
    
    raw_text = pytesseract.image_to_string(image)
    
    # add word-level data, includes bbox
    df = pytesseract.image_to_data(
        image,
        output_type=pytesseract.Output.DATAFRAME
    )

    df = df[(df["text"].notna()) & (df["text"].str.strip() != "")]
    df = df[df["conf"] > 0].reset_index(drop=True)
    
    return raw_text, df, image

In [30]:
def phase_constraints(raw_text, doc_category, fields, debug, output_dir, base_name):
    results = {}
    
    for engine_name, engine_path, prompt_fn in [
        ("engineA", ENGINE_A_PATH, build_constraints_prompt_A),
        ("engineB", ENGINE_B_PATH, build_constraints_prompt_B),
    ]:
        prompt = prompt_fn(raw_text, doc_category, fields)
        tokenizer, model = load_model_and_tokenizer(engine_path)
        
        response = run_chat_inference(
            model,
            tokenizer,
            prompt,
            max_new_tokens=800
        )
        
        release_model(model, tokenizer)
        maybe_save_text(response, f"{base_name}_{engine_name}_constraints.txt", debug, output_dir)
        
        results[engine_name] = response
        print(response)
        
    return results

In [31]:
def phase_extraction(raw_text, constraints, fields, debug, output_dir, base_name):
    results = {}
    
    engine_paths = {
        "engineA": ENGINE_A_PATH,
        "engineB": ENGINE_B_PATH
    }
    
    for engine_name, constraint_trace in constraints.items():
        prompt = build_extraction_prompt(raw_text, constraint_trace, fields, DOC_CATEGORY)
        
        tokenizer, model = load_model_and_tokenizer(engine_paths[engine_name])
        response = run_chat_inference(
            model,
            tokenizer,
            prompt,
            max_new_tokens=1600
        )
        
        release_model(model, tokenizer)
        
        extraction = json.loads(extract_json_block(response))
        # extraction = normalize_extraction_output(extraction, fields)

        maybe_save_json(extraction, f"{base_name}_{engine_name}_extraction.json", debug, output_dir)
        
        results[engine_name] = extraction
        print(extraction)
        
    return results

In [32]:
def phase_extraction(raw_text, constraints, fields, debug, output_dir, base_name):
    results = {}

    engine_paths = {
        "engineA": ENGINE_A_PATH,
        "engineB": ENGINE_B_PATH
    }

    for engine_name, constraint_trace in constraints.items():
        tokenizer, model = load_model_and_tokenizer(engine_paths[engine_name])
        engine_started_at = time.time()

        extraction = {
            "field_extraction": {},
            "evidence_trace": {},
            "reasoning": {},
            "engine_metrics": {
                "elapsed_seconds": 0.0,
                "prompt_tokens": 0,
                "completion_tokens": 0,
                "total_tokens": 0,
                "field_count": len(fields),
            }
        }

        for field in fields:
            prompt = build_single_extraction_prompt(
                raw_text,
                constraint_trace,
                field,
                DOC_CATEGORY
            )

            response, usage = run_chat_inference(
                model,
                tokenizer,
                prompt,
                max_new_tokens=500,
                return_usage=True
            )

            extraction["engine_metrics"]["prompt_tokens"] += usage["prompt_tokens"]
            extraction["engine_metrics"]["completion_tokens"] += usage["completion_tokens"]
            extraction["engine_metrics"]["total_tokens"] += usage["total_tokens"]

            try:
                parsed = json.loads(extract_json_block(response))

                extraction["field_extraction"][field] = parsed.get("field_extraction", "")
                extraction["evidence_trace"][field] = parsed.get("evidence_trace", "")
                extraction["reasoning"][field] = parsed.get(
                    "reasoning",
                    "No sufficiently supported value was found for this field."
                )

            except Exception as e:
                print(f"[{engine_name}] failed to parse field '{field}': {e}")
                print(response)

                extraction["field_extraction"][field] = ""
                extraction["evidence_trace"][field] = ""
                extraction["reasoning"][field] = "No sufficiently supported value was found for this field."

        extraction["engine_metrics"]["elapsed_seconds"] = round(time.time() - engine_started_at, 4)

        release_model(model, tokenizer)

        maybe_save_json(
            extraction,
            f"{base_name}_{engine_name}_extraction.json",
            debug,
            output_dir
        )

        results[engine_name] = extraction
        print(f"\n===== {engine_name} extraction =====")
        print(json.dumps(extraction, indent=2, ensure_ascii=False))

    return results

In [33]:
def phase_rule_synthesis(constraints, doc_category, fields, debug, output_dir, base_name):
    prompt = build_rule_synthesis_prompt(
        constraints["engineA"],
        constraints["engineB"],
        doc_category, fields
    )
    
    tokenizer, model = load_model_and_tokenizer(EVAL_MODEL_PATH)
    response = run_chat_inference(
        model,
        tokenizer,
        prompt,
        max_new_tokens=800
    )

    print(response)
    
    release_model(model, tokenizer)
    consolidated = json.loads(extract_json_block(response))
    
    maybe_save_json(consolidated, f"{base_name}_eval_constraints.json", debug, output_dir)
    
    return consolidated

In [34]:
def phase_self_justification(extractions, consolidated_rules, raw_text, doc_category, fields, debug, output_dir, base_name):
    results = {}
    
    for engine_name, extraction in extractions.items():
        engine_metrics = extraction.get("engine_metrics", {})
        engine_result = {
            "engine": engine_name,
            "engine_metrics": engine_metrics,
            "field_results": {}
        }
        
        for field in fields:
            extracted_value = extraction["field_extraction"][field]
            evidence_text = extraction["evidence_trace"][field]
            reasoning_text = extraction["reasoning"][field]
            rules = consolidated_rules["consolidated_rules"][field]
            sanity_result = run_sanity_check(extracted_value)

            prompt = build_stage2_prompt(
                field_name=field,
                rules=rules,
                extracted_value=extracted_value,
                evidence_text=evidence_text,
                reasoning_text=reasoning_text,
                ocr_raw_text=raw_text,
                doc_category=doc_category,
            )
            
            tokenizer, model = load_model_and_tokenizer(EVAL_MODEL_PATH)
            response = run_chat_inference(model, tokenizer, prompt, max_new_tokens=600)
            
            release_model(model, tokenizer)
            engine_result["field_results"][field] = {
                **json.loads(extract_json_block(response)),
                "engine_metrics": engine_metrics,
            }

        maybe_save_json(engine_result, f"{base_name}_{engine_name}_self_judge.json", debug, output_dir)
        results[engine_name] = engine_result
        
    return results

In [35]:
def _is_high_quality_candidate(field_result):
    return (
        bool(field_result.get("extracted_value"))
        and field_result.get("rule_consistency") == "high"
        and field_result.get("engine_self_consistency") == "strong"
        and field_result.get("ocr_alignment") == "strong"
        and field_result.get("ocr_corruption") == "absent"
    )

def _compare_engine_efficiency(engineA_metrics, engineB_metrics):
    a_tokens = engineA_metrics.get("total_tokens", float("inf"))
    b_tokens = engineB_metrics.get("total_tokens", float("inf"))
    a_time = engineA_metrics.get("elapsed_seconds", float("inf"))
    b_time = engineB_metrics.get("elapsed_seconds", float("inf"))

    a_not_worse = a_tokens <= b_tokens and a_time <= b_time
    b_not_worse = b_tokens <= a_tokens and b_time <= a_time
    a_strictly_better = a_tokens < b_tokens or a_time < b_time
    b_strictly_better = b_tokens < a_tokens or b_time < a_time

    if a_not_worse and a_strictly_better:
        return "engineA"
    if b_not_worse and b_strictly_better:
        return "engineB"
    return None

def phase_cross_judgment(self_justifications, extractions, consolidated_rules, doc_category, fields, debug, output_dir, base_name):
    stage3 = {
        "field_results": {}
    }
    
    for field in fields:
        engineA_result = self_justifications["engineA"]["field_results"][field]
        engineB_result = self_justifications["engineB"]["field_results"][field]
        rules = consolidated_rules["consolidated_rules"][field]

        prompt = build_cross_judge_prompt(
            field, rules, engineA_result, engineB_result, doc_category
        )
        tokenizer, model = load_model_and_tokenizer(EVAL_MODEL_PATH)
        response = run_chat_inference(model, tokenizer, prompt, max_new_tokens=600)
        release_model(model, tokenizer)
        
        judged = json.loads(extract_json_block(response))

        if _is_high_quality_candidate(engineA_result) and _is_high_quality_candidate(engineB_result):
            better_engine = _compare_engine_efficiency(
                self_justifications["engineA"].get("engine_metrics", {}),
                self_justifications["engineB"].get("engine_metrics", {}),
            )
            if better_engine and judged.get("selected_engine") != better_engine:
                better_result = engineA_result if better_engine == "engineA" else engineB_result
                judged["selected_engine"] = better_engine
                judged["recommended_value"] = better_result.get("extracted_value", "")
                judged["selection_reason"] = (
                    "Both engines produced similarly strong extractions, so the more efficient engine was selected "
                    "as the secondary tiebreaker."
                )
                judged["final_rule_consistency"] = better_result.get("rule_consistency", judged.get("final_rule_consistency", ""))
                judged["final_engine_self_consistency"] = better_result.get("engine_self_consistency", judged.get("final_engine_self_consistency", ""))
                judged["final_ocr_alignment"] = better_result.get("ocr_alignment", judged.get("final_ocr_alignment", ""))
                judged["final_ocr_corruption"] = better_result.get("ocr_corruption", judged.get("final_ocr_corruption", ""))

        engineA_metrics = self_justifications["engineA"].get("engine_metrics", {})
        engineB_metrics = self_justifications["engineB"].get("engine_metrics", {})
        selected_engine = judged.get("selected_engine", "")
        selected_metrics = {}
        if selected_engine == "engineA":
            selected_metrics = engineA_metrics
        elif selected_engine == "engineB":
            selected_metrics = engineB_metrics

        stage3["field_results"][field] = {
            "recommended_value": judged.get("recommended_value", ""),
            "selected_engine": selected_engine,
            "selection_reason": judged.get("selection_reason", ""),
            "selected_engine_total_tokens": selected_metrics.get("total_tokens", 0),
            "selected_engine_elapsed_seconds": selected_metrics.get("elapsed_seconds", 0.0),
            "final_rule_consistency": judged.get("final_rule_consistency", ""),
            "final_engine_self_consistency": judged.get("final_engine_self_consistency", ""),
            "final_ocr_alignment": judged.get("final_ocr_alignment", ""),
            "final_ocr_corruption": judged.get("final_ocr_corruption", ""),
            "field_confidence": judged.get("field_confidence", ""),
            "field_state": judged.get("field_state", ""),
            "state_reason": judged.get("state_reason", ""),
            "engineA_evidence": extractions["engineA"]["evidence_trace"].get(field, ""),
            "engineB_evidence": extractions["engineB"]["evidence_trace"].get(field, ""),
        }


    maybe_save_json(stage3, f"{base_name}_cross_judge.json", debug, output_dir)
    
    return stage3

In [36]:
def phase_visualize(
    cross_result: dict,
    fields: list,
    output_dir: str,
    base_name: str,
    word_df,
    image,
    padding: int = 6,
):
    os.makedirs(output_dir, exist_ok=True)

    vis = image.convert("RGBA")
    overlay = Image.new("RGBA", vis.size, (255, 255, 255, 0))
    draw_overlay = ImageDraw.Draw(overlay)
    draw_main = ImageDraw.Draw(vis)

    font = ImageFont.load_default()
    font_small = ImageFont.load_default()

    field_colors = generate_field_colors(len(fields))
    field_results = cross_result.get("field_results", {})
    legend_items = []

    for idx, field in enumerate(fields):
        result = field_results.get(field, {})
        recommended_value = result.get("recommended_value", "")
        field_state = result.get("field_state", "fail")
        field_confidence = result.get("field_confidence", "low")

        evidence_text = ""
        for eng_key in ["engineA_evidence", "engineB_evidence"]:
            ev = result.get(eng_key, "")
            if ev:
                evidence_text = ev
                break
        if not evidence_text:
            evidence_text = recommended_value

        bbox = match_evidence_to_bboxes(evidence_text, word_df)

        color_rgb  = field_colors[idx]
        border_rgb = STATE_BORDER.get(field_state, (128, 128, 128))
        icon = STATE_ICON.get(field_state, "?")

        if bbox:
            l, t, r, b = bbox
            l, t, r, b = l - padding, t - padding, r + padding, b + padding

            draw_overlay.rectangle(
                [l, t, r, b],
                fill=(*color_rgb, 50)
            )
            for offset in range(3):
                draw_main.rectangle(
                    [l - offset, t - offset, r + offset, b + offset],
                    outline=(*border_rgb, 220),
                )

            label = f"{field}: {recommended_value}  {icon}"
            bbox_text = draw_main.textbbox((0, 0), label, font=font)
            tw = bbox_text[2] - bbox_text[0]
            th = bbox_text[3] - bbox_text[1]

            tag_x = l
            tag_y = max(0, t - th - 8)

            draw_main.rectangle(
                [tag_x, tag_y, tag_x + tw + 10, tag_y + th + 6],
                fill=(*border_rgb, 230)
            )
            draw_main.text(
                (tag_x + 5, tag_y + 3),
                label,
                font=font,
                fill=(255, 255, 255, 255)
            )

        legend_items.append((field, recommended_value, field_state, field_confidence, color_rgb, border_rgb, icon))

    vis = Image.alpha_composite(vis, overlay)
    vis = vis.convert("RGB")

    legend_width = 320
    legend_img = Image.new("RGB", (legend_width, vis.height), (245, 245, 245))
    ld = ImageDraw.Draw(legend_img)

    ld.text((12, 12), "Extraction Summary", font=font, fill=(30, 30, 30))
    ld.line([(8, 38), (legend_width - 8, 38)], fill=(180, 180, 180), width=1)

    y_cursor = 50
    for (field, value, state, confidence, color_rgb, border_rgb, icon) in legend_items:
        ld.rectangle([10, y_cursor, 26, y_cursor + 16], fill=color_rgb)
        ld.text((34, y_cursor), field, font=font, fill=(30, 30, 30))
        y_cursor += 22
        display_val = value if value else "(not found)"
        ld.text((34, y_cursor), display_val, font=font_small, fill=(60, 60, 60))
        y_cursor += 20
        state_label = f"{icon} {state}  |  conf: {confidence}"
        ld.text((34, y_cursor), state_label, font=font_small, fill=border_rgb)
        y_cursor += 28
        ld.line([(10, y_cursor), (legend_width - 10, y_cursor)], fill=(210, 210, 210), width=1)
        y_cursor += 10

    combined = Image.new("RGB", (vis.width + legend_width, vis.height), (255, 255, 255))
    combined.paste(vis, (0, 0))
    combined.paste(legend_img, (vis.width, 0))

    out_path = os.path.join(output_dir, f"{base_name}_annotated.jpg")
    combined.save(out_path, quality=95)
    print(f"[visualize] saved → {out_path}")
    return out_path

# Main Pipeline

In [37]:
import datetime

def run_pipeline(image_path, doc_category, fields, debug=False):

    pipeline_start = time.time()
    base_name  = os.path.splitext(os.path.basename(image_path))[0]
    output_dir = DEBUG_OUTPUT_DIR

    print("[1/6] OCR extraction...")
    raw_text, word_df, image = phase_ocr_with_bbox(image_path)   # ← 只跑一次 OCR
    maybe_save_text(raw_text, f"{base_name}_raw_text.txt", debug, output_dir)
    print(raw_text)
    print(word_df)

    print("[2/6] Constraint generation (Engine A & B)...")
    constraints = phase_constraints(raw_text, doc_category, fields, debug, output_dir, base_name)

    print("[3/6] Constraint-guided extraction (Engine A & B)...")
    extractions = phase_extraction(raw_text, constraints, fields, debug, output_dir, base_name)

    print("[4/6] Rule synthesis + self-justification...")
    consolidated_rules  = phase_rule_synthesis(constraints, doc_category, fields, debug, output_dir, base_name)
    self_justifications = phase_self_justification(
        extractions, consolidated_rules, raw_text, doc_category, fields, debug, output_dir, base_name
    )

    print("[5/6] Cross-engine judgment...")
    cross_result = phase_cross_judgment(
        self_justifications, extractions, consolidated_rules, doc_category, fields, debug, output_dir, base_name
    )

    print("[6/6] Provenance visualization...")
    vis_path = phase_visualize(
        cross_result = cross_result,
        fields       = fields,
        output_dir   = output_dir,
        base_name    = base_name,
        word_df      = word_df,
        image        = image,
    )

    final_result = {
        "metadata": {
            "image_path":       image_path,
            "doc_category":     doc_category,
            "fields":           fields,
            "debug":            debug,
            "timestamp":        datetime.datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
            "elapsed_seconds":  round(time.time() - pipeline_start, 1),
            "annotated_image":  vis_path,
        },
        "field_results": cross_result["field_results"],
    }

    maybe_save_json(final_result, f"{base_name}_final_result.json", debug, output_dir)
    print("\n=== FINAL RESULT ===")
    print(json.dumps(final_result, indent=2, ensure_ascii=False))

    return final_result

In [38]:
result = run_pipeline(
    image_path = IMAGE_PATH,
    doc_category = DOC_CATEGORY,
    fields = FIELDS,
    debug = DEBUG
)

[1/6] OCR extraction...
tan woon yann

BOOK TA -K (TAMAN DAYA) SDN BHD
TBO7-W
NO.5? $5,57 & 59, JALAN SAGU 18,
TAMAN DAYA
81100 JOHOR BAHRU.
JOHOR.

LAM MCAT

Document Ho : TDO1167104

Date 25/12/2018 8:13:39 PM
Cashier MANIS
Member
CASH BILL
CODE/DESC PRICE Dise AMOUNT
Quy RM RM
9556939040118 AF MODELLING CLAY KIDDY FiSHt
1PC + 9.00) 0,00 9.00
Total : 9,00
Rour ding Adjustment 0.00
Round. :d Total (RM): 9.00
Cash a 40.00.
CHANGE 00

GOODS SOLD ARE NOT RETURNAP
EXCHANGEABLE

THANK YOU
PLEASE COME AGAIN t

    level  page_num  block_num  par_num  line_num  word_num  left  top  width  \
0       5         1          1        1         1         1    75   32     51   
1       5         1          1        1         1         2   138   37     91   
2       5         1          1        1         1         3   241   37     78   
3       5         1          2        1         1         1    73   97     52   
4       5         1          2        1         1         2   132   97     23   
.. 

Loading weights: 100%|██████████| 291/291 [00:04<00:00, 68.26it/s, Materializing param=model.norm.weight]                              
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


```json
{
  "constraint_summary": {
    "company": [
      "Typically appears as a name with a mix of alphabetic characters, possibly with a suffix like 'SDN BHD'",
      "Often includes words like 'TA', 'SDN', or 'BHD' indicating a company name",
      "May be a multi-line text block"
    ],
    "date": [
      "Usually in the format 'DD/MM/YYYY' or 'DD/MM/YYYY HH:MM:SS'",
      "May include time in 24-hour format",
      "Often appears near the top of the document"
    ],
    "address": [
      "Typically includes a street number, street name, and postal code",
      "May include words like 'JALAN', 'TAMAN', or 'JOHOR' indicating a location",
      "Often appears on multiple lines"
    ],
    "total": [
      "Usually a numeric value with a currency symbol like 'RM'",
      "May include a decimal point and two digits after the decimal point",
      "Often appears near the bottom of the document"
    ],
    "phone number": [
      "Not present in this document, but typically a sequenc

Loading weights: 100%|██████████| 339/339 [00:04<00:00, 84.22it/s, Materializing param=model.norm.weight]                               
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


```json
{
  "constraint_summary": {
    "company": ["usually contains a business name", "often includes 'SDN BHD'", "appears near the top of the document", "may include alphanumeric characters and special characters like hyphens and parentheses"], 
    "date": ["formatted as day/month/year", "often includes time", "appears near the top or in the header section", "may include colons and slashes"], 
    "address": ["multi-line text block", "includes street numbers, names, and city/town", "appears below the company name", "contains alphanumeric characters and special characters like commas and periods"], 
    "total": ["numeric value with currency symbol 'RM'", "ends with a total amount", "appears near the bottom of the document", "may include decimal points"], 
    "phone number": ["numeric value", "often includes country code or area code", "may appear in the address or contact section", "can be formatted with spaces or hyphens"]
  }
}
```
[3/6] Constraint-guided extraction (Engine A & 

...

{
  "consolidated_rules": {
    "company": [
      "Typically appears as a name with a mix of alphabetic characters, possibly with a suffix like 'SDN BHD' or 'Sdn Bhd'",
      "Often includes words like 'TA', 'SDN', or 'BHD' indicating a company name",
      "May be a multi-line text block and may include alphanumeric characters and special characters like hyphens and parentheses"
    ],
    "date": [
      "Usually in the format 'DD/MM/YYYY' or 'DD/MM/YYYY HH:MM:SS', may include time in 24-hour format",
      "Often appears near the top of the document or in the header section, and may include colons and slashes"
    ],
    "address": [
      "Typically includes a street number, street name, and postal code",
      "May include words like 'JALAN', 'TAMAN', or 'JOHOR' indicating a location",
      "Often appears below the company name, on multiple lines, and contains alphanumeric characters and special characters like commas and periods"
    ],
    "total": [
      "Usually a numeric v

Loading weights: 100%|██████████| 291/291 [00:03<00:00, 74.99it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 72.90it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 78.96it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 73.42it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 74.90it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for o

[5/6] Cross-engine judgment...


Loading weights: 100%|██████████| 291/291 [00:03<00:00, 75.45it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 72.98it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 77.16it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading weights: 100%|██████████| 291/291 [00:04<00:00, 72.56it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 74.19it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:2 for o

[6/6] Provenance visualization...
[visualize] saved → data/temp/debug/20260324_194445/X00016469612_annotated.jpg

=== FINAL RESULT ===
{
  "metadata": {
    "image_path": "data/temp/X00016469612.jpg",
    "doc_category": "receipt",
    "fields": [
      "company",
      "date",
      "address",
      "total",
      "phone number"
    ],
    "debug": true,
    "timestamp": "2026-03-24T19:47:56",
    "elapsed_seconds": 190.5,
    "annotated_image": "data/temp/debug/20260324_194445/X00016469612_annotated.jpg"
  },
  "field_results": {
    "company": {
      "recommended_value": "TAMAN DAYA SDN BHD",
      "selected_engine": "engineB",
      "selection_reason": "Both engines produced similarly strong extractions, so the more efficient engine was selected as the secondary tiebreaker.",
      "selected_engine_total_tokens": 4657,
      "selected_engine_elapsed_seconds": 10.7885,
      "final_rule_consistency": "high",
      "final_engine_self_consistency": "strong",
      "final_ocr_alignmen